# Timings of application

In [1]:
%load_ext autoreload
%autoreload 2

import gc
import numpy as np
import pickle
from time import time
import matplotlib.pyplot as plt
from figure_manager import FigureManager
from plots import *


import DynamicTimeAllocationModel

path = 'output/'
plt.rcParams.update({'font.size': 14})

fm = FigureManager(path, use_latex=False)

# c++ settings
do_compile = True
load_par = True
threads = 128

# from EconModel import cpptools
# cpptools.setup_nlopt(folder='cppfuncs/', do_print=True)

## Monte Carlo Settings

In [2]:
MC_num = 1 #200 # number of Monte Carlo simulations
C_num_grid = (25,50,75,100,200) # number of grid points in consumption grid i iEGM
do_print = True # whether to print results during Monte Carlo iterations

## Illustrate the pre-computation of consumption interpolator

In [3]:
filename = 'calibrated_par'
with open(f"par_files/{filename}.pkl", "rb") as f:
    settings = pickle.load(f)

## Solve the "true" model using many grid points in VFI

In [4]:
# overwrite settings with specs_true
settings_true = settings.copy()

settings_true['num_multistart'] = 0
settings_true['do_egm'] = False
settings_true['interp_method'] = 'linear'
settings_true['interp_inverse'] = True
settings_true['precompute_intratemporal'] = True
settings_true['centered_gradient'] = True

settings_true['num_A'] = 50
settings_true['num_A_pd'] = 50
settings_true['num_K'] = 12
settings_true['num_power'] = 15
settings_true['num_love'] = 15
settings_true['num_types'] = 4
settings_true['num_Ctot'] = 1024
settings_true['num_marg_u'] = 1024

model_true = DynamicTimeAllocationModel.HouseholdModelClass(par=settings_true)
model_true.link_to_cpp(force_compile=True)

In [5]:
stop

NameError: name 'stop' is not defined

In [ ]:
%time model_true.solve()
%time model_true.simulate()
# model_true.cpp.delink()

CPU times: total: 28d 18h 57min 28s
Wall time: 6h 9min 21s
CPU times: total: 2.03 s
Wall time: 104 ms


In [ ]:
print(model_true.sim.mean_lifetime_util[0])

180.76640050400496


## Accuracy measures

In [ ]:
model_specs = {    
    'iegm': {'table_name':'iEGM', 'do_egm': True, 'interp_method': 'linear'},
    'iegm (num)': {'table_name':'EGM (num)', 'do_egm': True, 'interp_method': 'numerical'},
    'vfi': {'table_name':'VFI', 'do_egm': False},
}


In [ ]:
models = {}
results = {}
for name_base,specs in model_specs.items():
# for name_base,specs in {**model_specs, **model_specs_pre}.items():
    
    
    if ('iegm' in name_base) and (specs['interp_method']!='numerical'):
        C_num_grid_now = C_num_grid
    else:
        C_num_grid_now = (20,) # not used
    
    for num_C in C_num_grid_now:  
        name = name_base + (f' num_C={num_C}' if len(C_num_grid_now)>1 else '')
        key = num_C if len(C_num_grid_now)>1 else ''
        
        if name not in results:
            results[name] = {}
        results[name][key] = {
            'timing':np.nan+np.ones(MC_num),
            'timing_rel':np.nan+np.ones(MC_num),
            'MSE':np.nan+np.ones(MC_num),
            'lw_MAD':np.nan+np.ones(MC_num),
            'lm_MAD':np.nan+np.ones(MC_num),
            'divorce_MAD':np.nan+np.ones(MC_num),
            'C_MAD':np.nan+np.ones(MC_num),
            'util_diff_pct':np.nan+np.ones(MC_num),
            'comp':np.nan+np.ones(MC_num),
            'lifetime_util':np.nan+np.ones(MC_num),
        } 
        
        if do_print: print(name,key)
        model_name = name
        # setup model and solution containers
        models[model_name] = DynamicTimeAllocationModel.HouseholdModelClass(par={**settings,**specs,'num_Ctot':num_C, 'num_marg_u': num_C})
        
        
        # loop over Monte Carlo iterations
        for i_mc in range(MC_num):
            if do_print: print(f'{i_mc+1}/{MC_num} running...')
            
            # re-simulate true model for this Monte Carlo iteration to get new draws
            true = model_true
            
            true.par.seed = i_mc
            true.draw_shocks()
            true.link_to_cpp(force_compile=False)
            true.simulate()
            # true.cpp.delink()
            
            # solution time
            models[model_name].link_to_cpp(force_compile=False)
            t0 = time()
            models[model_name].solve()
            secs = time() - t0
            if do_print: print(f'Solution time: {secs:.2f} seconds')
            
            # accuracy
            # MSE,MAD = models[model_name].MSE_consumption(true,grid_m_MSE)
            # if do_print: print(f'MSE in consumption function: {MSE:.4f}')
            # if do_print: print(f'MAD in consumption function: {MAD:.4f}')
            
            models[model_name].par.seed = i_mc
            models[model_name].draw_shocks()
            models[model_name].simulate()
            # models[model_name].cpp.delink()
            
            # diff_pct = models[model_name].lifetime_utility(util,true)
            # if do_print: print(f'Difference in lifetime utility (compared to true model): {diff_pct:.4f}')
            
            lw_MAD, lm_MAD, divorce_MAD, C_MAD = models[model_name].MAD_consumption(true)
            if do_print: print(f'MAD in labor_w function: {lw_MAD:.4f}')
            if do_print: print(f'MAD in labor_M function: {lm_MAD:.4f}')
            if do_print: print(f'MAD in divorce: {divorce_MAD:.4f}')
            if do_print: print(f'MAD in consumption function: {C_MAD:.4f}')
            
            lifetime_util = models[model_name].sim.mean_lifetime_util[0]
            if do_print: print(f'Lifetime utility: {lifetime_util:.12f}')
            
            comp = models[model_name].wealth_compensation(true)
            if do_print: print(f'Wealth compensation (pct of household exp. income): {comp:.4f}', end='\n\n')
            
            
            results[name][key]['timing'][i_mc] = secs
            # results[name][key]['timing_rel'][i_mc] = secs/results['vfi'][restricted_str]['timing'][i_mc]
            # results[name][key]['MSE'][i_mc] = MSE
            results[name][key]['lw_MAD'][i_mc] = lw_MAD
            results[name][key]['lm_MAD'][i_mc] = lm_MAD
            results[name][key]['divorce_MAD'][i_mc] = divorce_MAD
            results[name][key]['C_MAD'][i_mc] = C_MAD
            # results[name][key]['util_diff_pct'][i_mc] = diff_pct
            results[name][key]['lifetime_util'][i_mc] = lifetime_util
            results[name][key]['comp'][i_mc] = comp
            
            # delete to free memory
            del models[model_name]
            gc.collect()

iegm num_C=25 25
1/1 running...
Solution time: 108.34 seconds
MAD in labor_w function: 0.0830
MAD in labor_M function: 0.0888
MAD in divorce: 0.0000
MAD in consumption function: 2.8535
Lifetime utility: 167.704404151357
Wealth compensation (pct of household exp. income): 134.0449

iegm num_C=50 50
1/1 running...
Solution time: 109.90 seconds
MAD in labor_w function: 0.0297
MAD in labor_M function: 0.0336
MAD in divorce: 0.0000
MAD in consumption function: 1.7255
Lifetime utility: 176.494695544467
Wealth compensation (pct of household exp. income): 35.8798

iegm num_C=75 75
1/1 running...
Solution time: 110.21 seconds
MAD in labor_w function: 0.0189
MAD in labor_M function: 0.0231
MAD in divorce: 0.0000
MAD in consumption function: 1.5828
Lifetime utility: 178.343480192884
Wealth compensation (pct of household exp. income): 22.4434

iegm num_C=100 100
1/1 running...
Solution time: 110.24 seconds
MAD in labor_w function: 0.0186
MAD in labor_M function: 0.0211
MAD in divorce: 0.0000
MAD i

In [ ]:
# write to latex file
table_path = 'output/MC_limited'
output = ('timing','timing_rel','lw_MAD', 'lm_MAD', 'divorce_MAD', 'C_MAD','comp')
output_labels = '& time (s) & rel. & MAD(lw) & MAD(lm) & MAD(divorce) & MAD(C)  & Comp(A) '

num_cols_per_model = len(output)

for pre in (False,):
    pre_str = ', pre' if pre else ''
    tab_name = table_path + '_pre' if pre else table_path
    
    num_models = 1 
    num_cols = 2 + num_cols_per_model*num_models
    
    with open(tab_name+'.tex','w') as f:
        # table head
        f.write('\\begin{tabular}{ll' + 'c'* (num_cols-2) + '} \\toprule\n')
        f.write(f'&& \\multicolumn{{{num_cols_per_model}}}{{c}}{{Application model}}  \\\\ \n ')
        f.write(f'\\cmidrule(lr){{3-{2+num_cols_per_model}}} &')
        for _ in range(num_models): f.write(output_labels)
        f.write('\\\\ \\midrule \n')
        
        # table body
        # model_specs_now = model_specs_pre if pre else model_specs
        model_specs_now = model_specs
        for name_base,specs in reversed(model_specs_now.items()):
            table_name = specs['table_name']
            
            if ('iegm' in name_base) and (specs['interp_method']!='numerical'):
                C_num_grid_now = C_num_grid
            else:
                C_num_grid_now = (20,) # not used
            
            #num_multi = 1 if len(C_num_grid_now)>1 else 2
            f.write(f'\\multicolumn{{{2}}}{{l}}{{{table_name}}} ')
            for iC,num_C in enumerate(C_num_grid_now):  
                if len(C_num_grid_now)>1:
                    if iC==0: # skip entire row
                        f.write(f'\\multicolumn{{{num_cols-2}}}{{c}}{{}} \\\\ \n')
                    
                    f.write(f'& $\\#_C$={num_C}')
                
                name = name_base + (f' num_C={num_C}' if len(C_num_grid_now)>1 else '')
                key = num_C if len(C_num_grid_now)>1 else ''
                
                for out in output:
                    if out == 'timing_rel':
                        avg = np.mean(results[name][key]['timing'])/np.mean(results['vfi']['']['timing'])
                    else:
                        avg = np.mean(results[name][key][out])
                    if np.isnan(avg):
                        f.write('& N.A. ')
                    else:
                        f.write(f'& {avg:.3f} ')
                f.write('\\\\ \n')
        f.write('\\bottomrule \\end{tabular}\n')
        
        
    # output the table here by looping through all lines in the file and printing them
    with open(tab_name+'.tex','r') as f:
        for line in f:
            print(line,end='')  



\begin{tabular}{llccccccc} \toprule
&& \multicolumn{7}{c}{Application model}  \\ 
 \cmidrule(lr){3-9} && time (s) & rel. & MAD(lw) & MAD(lm) & MAD(divorce) & MAD(C)  & Comp(A) \\ \midrule 
\multicolumn{2}{l}{VFI} & 5355.287 & 1.000 & 0.043 & 0.038 & 0.000 & 1.743 & 126.786 \\ 
\multicolumn{2}{l}{EGM (num)} & 664.681 & 0.124 & 0.043 & 0.039 & 0.000 & 1.747 & 126.650 \\ 
\multicolumn{2}{l}{iEGM} \multicolumn{7}{c}{} \\ 
& $\#_C$=25& 108.335 & 0.020 & 0.083 & 0.089 & 0.000 & 2.853 & 134.045 \\ 
& $\#_C$=50& 109.900 & 0.021 & 0.030 & 0.034 & 0.000 & 1.725 & 35.880 \\ 
& $\#_C$=75& 110.206 & 0.021 & 0.019 & 0.023 & 0.000 & 1.583 & 22.443 \\ 
& $\#_C$=100& 110.236 & 0.021 & 0.019 & 0.021 & 0.000 & 1.561 & 19.025 \\ 
& $\#_C$=200& 158.848 & 0.030 & 0.014 & 0.017 & 0.000 & 1.525 & 17.325 \\ 
\bottomrule \end{tabular}


In [ ]:
stop

NameError: name 'stop' is not defined

In [ ]:
# models

In [ ]:
plot_names = ('vfi, pre','iegm (num), pre','iegm num_C=25','iegm num_C=50')
for t in (-20,):
    fig,ax = plt.subplots()
    for name in plot_names:
        m_interp = np.concatenate((np.array([0.0]),models[name].sol.M[t]))
        c_interp = np.concatenate((np.array([0.0]),models[name].sol.C[t]))
        ax.plot(m_interp,c_interp,label=name,marker='.')
        ax.legend();
        ax.set(title=f'period {models[name].par.T+t+1} (of {models[name].par.T})');

In [ ]:
for t in (-5,-20):
    fig,ax = plt.subplots()
    for name,specs in model_specs_restricted.items():
        ax.scatter(models[name].sol.M[t,:],models[name].sol.C[t,:],label=name)
        ax.legend();
        ax.set(title=f'period {models[name].par.T+t+1} (of {models[name].par.T})');

In [ ]:
stop

In [ ]:
test_specs = {
    
    'num_m': 100,
    'num_C':50,
    
}


In [ ]:
# solve models with different approaches
models = dict()
for restricted in (True,False):
    restricted_str = 'unrestricted model' if not restricted else "restricted model"
    print(f'***** {restricted_str} *****')
    for method in ('vfi','egm','iegm'):
        for interp_method in ('linear',):
        # for interp_method in ('linear','numerical'):
            interp_str = ' num' if (interp_method=='numerical') & (method == 'iegm') else ''
            if (interp_method=='numerical') & (method != 'iegm'):
                continue
            for precomputed in (False,True):
                precomputed_str = '' if not precomputed  else ' (precomputed intra)'
                if (method=='egm') & (not restricted):
                    continue
                # if (method != 'iegm') & (precomputed):
                #     continue
                print(method+interp_str+precomputed_str)
                model_name = method+restricted_str+precomputed_str+interp_str
                models[model_name] = UnitaryModelClass(par={**test_specs,'method':method,'restricted_model':restricted,'precompute_intra':precomputed,'interp_method':interp_method})
                %time models[model_name].solve()
                print('\n',end='')

In [ ]:
# Q: why is the precompute_intra so important? Also for iEGM? is it the last-period and the calculation after finding Ctot in general?
# A: The use of the marginal utility in the Euler equation requires many re-calucations of the intra-temporal problem. Precomputing it is obious.
# But then what about the numerical EGM? Should that not pre-compute the marginal util?

# there is something happening with the interpolation over different grids. If I try to use the precomputed marginal_HH_util  and interpolate, then there is a problem elsewhere. Is it extrapolation that leads to negative marginal utility...?

In [ ]:
for restricted in (True,False):
    restricted_str = '' if not restricted else ", restricted"
    for t in (-5,-20):
        fig,ax = plt.subplots()
        for method in ('vfi','egm','iegm'):
            for interp_method in ('linear',):
            # for interp_method in ('linear','numerical'):
                interp_str = 'num' if (interp_method=='numerical') & (method == 'iegm') else ''
                if (interp_method=='numerical') & (method != 'iegm'):
                    continue
                for precomputed in (False,True):
                    precomputed_str = '' if not precomputed  else ' (precomputed)'
                    if (method=='egm') & (not restricted):
                        continue
                    if (method != 'iegm') & (precomputed):
                        continue
                    
                    model_name = method+restricted_str+precomputed_str+interp_str
                    ax.scatter(models[model_name].sol.M[t,:],models[model_name].sol.C[t,:],label=model_name+precomputed_str)
                    ax.legend();
                    ax.set(title=f'period {models[model_name].par.T+t+1} (of {models[model_name].par.T})'+restricted_str);

In [ ]:
stop
# all the monte carlo stuff is not implemented yet.

In [ ]:
# model_true = UnitaryModelClass({par=**true_specs'num_m:300})
# model_true.solve()

Monte Carlo Runs

In [ ]:
PRINT = False
# setup Monte Carlo results containers
timing = {
    'vfi':np.nan + np.zeros(MC_num),
    'egm':np.nan + np.zeros(MC_num),
    'egm, numerical':np.nan + np.zeros(MC_num),

    'iegm, linear':dict(),
    
}
mse = {
    'vfi':np.nan + np.zeros(MC_num),
    'egm':np.nan + np.zeros(MC_num),
    'egm, numerical':np.nan + np.zeros(MC_num),

    'iegm, linear':dict(),
}
util = {
    'vfi':np.nan + np.zeros(MC_num),
    'egm':np.nan + np.zeros(MC_num),
    'egm, numerical':np.nan + np.zeros(MC_num),
    
    'iegm, linear':dict(),
}
for i_c,num_C in enumerate(C_num_grid):
    timing['iegm, linear'][num_C] = np.nan + np.zeros(MC_num)
    mse['iegm, linear'][num_C] = np.nan + np.zeros(MC_num)
    util['iegm, linear'][num_C] = np.nan + np.zeros(MC_num)

# loop over Monte Carlo simulations
for i_mc in range(MC_num):
    if PRINT: print(f'{i_mc+1}/{MC_num} running...')
    # simulate true model (solved once above)
    model_true.par.seed = i_mc
    model_true.allocate()
    model_true.simulate()
    model_true.lifetime_utility(HH_util) 
    true_mean_lifetime_util = model_true.sim.mean_lifetime_util

    # setup alternative model solutios
    model = UnitaryModelClass(par=specs)
    model.par.seed = i_mc
    model.allocate()

    # VFI and EGM
    for method in ['vfi','egm']:
        model.par.method = method
        
        # Timing
        t0 = time()
        model.solve()
        timing[model.par.method][i_mc] = time() - t0

        # Accuracy
        model.simulate()
        mse[model.par.method][i_mc] = model.sim.mean_log10_euler
        util[model.par.method][i_mc] = np.abs((model.sim.mean_lifetime_util - true_mean_lifetime_util)/true_mean_lifetime_util) * 100

    # iEGM
    model.par.method = 'iegm'
    for interp_method in ('linear',):
        for interp_inverse in (False,True):
            model.par.interp_method = interp_method
            model.par.interp_inverse = interp_inverse
            method = f'{model.par.method}, {interp_method} inverse' if interp_inverse else f'{model.par.method}, {interp_method}'
            for i_c,num_C in enumerate(C_num_grid):
                model.par.num_C = num_C
                model.allocate()

                # Timing
                t0 = time()
                model.solve()
                timing[method][num_C][i_mc] = time() - t0

                # Accuracy
                model.simulate()
                mse[method][num_C][i_mc] = model.sim.mean_log10_euler
                util[method][num_C][i_mc] = np.abs((model.sim.mean_lifetime_util - true_mean_lifetime_util)/true_mean_lifetime_util) * 100

    model.par.interp_method = 'numerical'
    model.par.interp_inverse = False
    method = f'egm, {model.par.interp_method}'
    # Timing
    t0 = time()
    model.solve()
    timing[method][i_mc] = time() - t0

    # Accuracy
    model.simulate()
    mse[method][i_mc] = model.sim.mean_log10_euler
    util[method][i_mc] = np.abs((model.sim.mean_lifetime_util - true_mean_lifetime_util)/true_mean_lifetime_util) * 100


In [ ]:
print('lifetime util & Timing (rel. to VFI)')
timing_vfi = np.mean(timing['vfi'])
for method in ('vfi','egm','egm, numerical'):
    util_now = np.mean(util[method])
    time_now = np.mean(timing[method]) / timing_vfi
    print(f'{method}: {util_now:2.3f} & {time_now:2.3f} ')

for method in ('iegm, linear','iegm, linear inverse'):
    print(f'{method}: ')
    for i_c,num_C in enumerate(C_num_grid):
        util_now = np.mean(util[method][num_C]) 
        time_now = np.mean(timing[method][num_C]) / timing_vfi
        print(f'{num_C:d} {util_now:2.3f} & {time_now:2.3f} ')